## REAL DATABASE PROJECT (SQL + BUSINESS THINKING)
### GOAL

👉 Today you will:
- Work like a real Data Analyst / Backend Engineer
- Design a database
- Write real-world queries
- Build logic used in companies

### CUSTOMER ORDER MANAGEMENT SYSTEM (SQL Project)

### STEP 1 — Import Libraries

In [1]:
import sqlite3
import pandas as pd

### STEP 2 — Create Database

In [3]:
conn = sqlite3.connect("company.db")
cursor = conn.cursor()

This creates a real mini database file.
### STEP 3 — Create Tables (Database Design)
**Customers Table**

In [4]:
cursor.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    name TEXT,
    city TEXT
)
""")

**Products Table**

In [5]:
cursor.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name TEXT,
    price INT
)
""")

### Orders Table (connects customers + products)

In [6]:
cursor.execute("""
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    product_id INT,
    quantity INT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
""")

#### Why 3 tables?

This is relational database design
- No duplicate data
- Easy scaling
- Fast queries

### STEP 4 — Insert Sample Data

In [17]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Drop tables if they exist (clean slate)
cursor.execute("DROP TABLE IF EXISTS orders")
cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute("DROP TABLE IF EXISTS products")

# Create customers table
cursor.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    name TEXT,
    city TEXT
)
""")

# Create products table
cursor.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name TEXT,
    price INT
)
""")

# Create orders table
cursor.execute("""
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    product_id INT,
    quantity INT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
""")

# Now insert data (works fine!)
cursor.executemany("INSERT INTO customers VALUES (?,?,?)", [
    (1, "Ali", "Lahore"),
    (2, "Sara", "Karachi"),
    (3, "Ahmed", "Islamabad")   # inactive customer (no orders)
])

cursor.executemany("INSERT INTO products VALUES (?,?,?)", [
    (101, "Laptop", 100000),
    (102, "Mobile", 50000),
    (103, "Tablet", 30000)
])

cursor.executemany("INSERT INTO orders VALUES (?,?,?,?)", [
    (1, 1, 101, 1),
    (2, 2, 102, 2),
    (3, 1, 103, 1)
])

conn.commit()

### Verify data
print("Customers:")
print(pd.read_sql_query("SELECT * FROM customers", conn))
print("\nProducts:")
print(pd.read_sql_query("SELECT * FROM products", conn))
print("\nOrders:")
print(pd.read_sql_query("SELECT * FROM orders", conn))###

Customers:
   customer_id   name       city
0            1    Ali     Lahore
1            2   Sara    Karachi
2            3  Ahmed  Islamabad

Products:
   product_id product_name   price
0         101       Laptop  100000
1         102       Mobile   50000
2         103       Tablet   30000

Orders:
   order_id  customer_id  product_id  quantity
0         1            1         101         1
1         2            2         102         2
2         3            1         103         1


## REAL QUERIES
### QUERY 1 — Show ALL Orders (Full Details)

In [18]:
query = """
SELECT customers.name, products.product_name, orders.quantity,
(products.price * orders.quantity) AS total_price
FROM orders
JOIN customers ON orders.customer_id = customers.customer_id
JOIN products ON orders.product_id = products.product_id
"""
pd.read_sql_query(query, conn)

,name,product_name,quantity,total_price
0,Ali,Laptop,1,100000
1,Sara,Mobile,2,100000
2,Ali,Tablet,1,30000


**What happens?**

We JOIN 3 tables to see full order info.

**Result shows:**

Customer → Product → Quantity → Total price
### QUERY 2 — Customer Spending

In [19]:
query = """
SELECT customers.name,
SUM(products.price * orders.quantity) AS total_spent
FROM orders
JOIN customers ON orders.customer_id = customers.customer_id
JOIN products ON orders.product_id = products.product_id
GROUP BY customers.name
"""
pd.read_sql_query(query, conn)

,name,total_spent
0,Ali,130000
1,Sara,100000


Business meaning

How much money each customer gave to company.
### QUERY 3 — TOP CUSTOMER

In [20]:
query = """
SELECT customers.name,
SUM(products.price * orders.quantity) AS total_spent
FROM orders
JOIN customers ON orders.customer_id = customers.customer_id
JOIN products ON orders.product_id = products.product_id
GROUP BY customers.name
ORDER BY total_spent DESC
LIMIT 1
"""
pd.read_sql_query(query, conn)

,name,total_spent
0,Ali,130000


Business meaning

Find VIP customer

### QUERY 4 — MOST SOLD PRODUCT

In [21]:
query = """
SELECT products.product_name,
SUM(orders.quantity) AS total_sold
FROM orders
JOIN products ON orders.product_id = products.product_id
GROUP BY products.product_name
ORDER BY total_sold DESC
LIMIT 1
"""
pd.read_sql_query(query, conn)

,product_name,total_sold
0,Mobile,2


Business meaning

Which product sells most
### QUERY 5 — CITY-WISE REVENUE

In [22]:
query = """
SELECT customers.city,
SUM(products.price * orders.quantity) AS revenue
FROM orders
JOIN customers ON orders.customer_id = customers.customer_id
JOIN products ON orders.product_id = products.product_id
GROUP BY customers.city
"""
pd.read_sql_query(query, conn)

,city,revenue
0,Karachi,100000
1,Lahore,130000


Business meaning

Which city generates most money.
### QUERY 6 — INACTIVE CUSTOMERS (No Orders)

In [23]:
query = """
SELECT customers.name
FROM customers
LEFT JOIN orders
ON customers.customer_id = orders.customer_id
WHERE orders.order_id IS NULL
"""
pd.read_sql_query(query, conn)

,name
0,Ahmed


Business meaning

Customers who never purchased → marketing target

### Pandas vs SQL — Same Logic, Different Tool
In companies, data is stored in multiple tables like:

Customers | Orders | Products

We want answers like:
- Who spent the most?
- Which product sells the most?
- City-wise revenue?

We can solve this using Pandas (Python) or SQL (Database).

👉 The thinking is the same, only the tool changes.

### 1. Combine Tables
Pandas → merge()

SQL → JOIN

Goal: Combine customers + orders

Pandas
``` merged = pd.merge(customers, orders, on="customer_id")```
SQL
```
SELECT *
FROM customers
JOIN orders
ON customers.customer_id = orders.customer_id;
```
👉 Both mean: Combine tables using a common column

### 2. Group Data
Pandas → groupby()
SQL → GROUP BY

Goal: Total spending per customer

Pandas

`merged.groupby("name")["amount"].sum()`

SQL
```
SELECT name, SUM(amount)
FROM merged_table
GROUP BY name;
```
👉 Both mean: Make groups and calculate per group

### 3. Calculate Totals
Pandas → sum()
SQL → SUM()

Goal: Total revenue

Pandas
```
df["revenue"] = df["price"] * df["quantity"]
df["revenue"].sum()
```
SQL
```
SELECT SUM(price * quantity) FROM sales;
```
👉 Both mean: Add all values

# Pandas and SQL do the same work.

| **Pandas**    | **SQL**       | **Purpose**         |
|---------------|---------------|---------------------|
| merge()       | JOIN          | Combine tables      |
| groupby()     | GROUP BY      | Aggregate data      |
| sum()         | SUM()         | Calculate metrics   |

---

- Same logic → Different tools